In [ ]:
import sys
sys.path.append("..")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime, timedelta

from app.continual_learning.drift_detector import DriftDetector, ADWINDetector
from app.models.ensemble import EnsembleScores, DEFAULT_WEIGHTS

## Simulate a Concept Drift Scenario
# Phase 1: Stable fraud pattern (low amounts, known devices)
# Phase 2: New fraud pattern emerges (high velocity, new devices)

In [ ]:
np.random.seed(42)
N_STABLE     = 300
N_DRIFT      = 200

stable_amounts     = np.random.normal(0.05, 0.01, N_STABLE)   # Small normalized amounts
drift_amounts      = np.random.normal(0.40, 0.10, N_DRIFT)    # Suddenly large amounts
all_amounts        = np.concatenate([stable_amounts, drift_amounts])
drift_point        = N_STABLE

print(f"Simulating {N_STABLE + N_DRIFT} transactions")
print(f"Drift point: transaction #{drift_point}")

In [ ]:
detector     = ADWINDetector(delta=0.002)
drift_points = []

for i, val in enumerate(all_amounts):
    drifted = detector.update(float(np.clip(val, 0, 1)))
    if drifted:
        drift_points.append(i)
        print(f"  ⚠️  Drift detected at transaction #{i}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Top: raw feature values
axes[0].plot(range(len(all_amounts)), all_amounts, color="#7F8C8D", alpha=0.6, linewidth=0.8)
axes[0].axvline(x=drift_point, color="#E74C3C", linestyle="--", linewidth=2, label="True drift point")
for dp in drift_points:
    axes[0].axvline(x=dp, color="#F39C12", linestyle=":", linewidth=1.5, alpha=0.8)
axes[0].set_ylabel("Normalized Amount")
axes[0].set_title("Transaction Feature Values with Concept Drift")
axes[0].legend()

# Bottom: rolling mean to show distributional shift
window    = 30
roll_mean = [all_amounts[max(0, i-window):i+1].mean() for i in range(len(all_amounts))]
axes[1].plot(roll_mean, color="#2980B9", linewidth=1.5, label="Rolling mean (30-txn window)")
axes[1].axvline(x=drift_point, color="#E74C3C", linestyle="--", linewidth=2)
for dp in drift_points:
    axes[1].axvline(x=dp, color="#F39C12", linestyle=":", linewidth=1.5, alpha=0.8)
axes[1].set_xlabel("Transaction #")
axes[1].set_ylabel("Rolling Mean")
axes[1].set_title("Distribution Shift: Rolling Mean")

detected_patch = mpatches.Patch(color="#F39C12", label=f"ADWIN detected ({len(drift_points)} events)")
true_patch     = mpatches.Patch(color="#E74C3C", label="True drift point")
axes[1].legend(handles=[true_patch, detected_patch])

plt.tight_layout()
plt.savefig("drift_detection.png", dpi=150)
plt.show()

In [ ]:
full_detector = DriftDetector(delta=0.002)

def make_scores(unified: float) -> EnsembleScores:
    return EnsembleScores(
        transformer_score=unified * 0.9,
        graphsage_score=unified * 1.05,
        cnn_gnn_score=unified * 0.85,
        tssgc_score=unified * 0.95,
        gan_autoencoder_score=unified * 0.8,
        unified_score=unified,
        weights=DEFAULT_WEIGHTS,
    )

uncertainty_flags = []
drift_events_full = []

for i, amount in enumerate(all_amounts):
    body = {
        "amount": int(np.clip(amount, 0, 1) * 10_000_000),
        "is_new_device": amount > 0.3,
        "is_new_recipient": amount > 0.2,
        "currency": "NGN",
    }
    score_val = float(np.clip(amount * 1.5, 0, 1))
    scores    = make_scores(score_val)

    events = full_detector.observe(body, scores)
    if events:
        drift_events_full.extend(events)

    if full_detector.is_uncertain(scores):
        uncertainty_flags.append(i)

print(f"\nDrift Detector Summary:")
print(f"  Total drift events:   {full_detector.drift_count}")
print(f"  Uncertain predictions: {len(uncertainty_flags)}")
print(f"  Uncertainty rate:      {len(uncertainty_flags)/len(all_amounts):.2%}")


In [ ]:
score_values = [float(np.clip(a * 1.5, 0, 1)) for a in all_amounts]

plt.figure(figsize=(14, 5))
colors = ["#F39C12" if i in uncertainty_flags else "#2ECC71" if s < 0.65
          else "#E74C3C" if s >= 0.90 else "#3498DB"
          for i, s in enumerate(score_values)]

plt.scatter(range(len(score_values)), score_values, c=colors, s=10, alpha=0.7)
plt.axhline(y=0.65, color="#3498DB", linestyle="--", linewidth=1.5, label="Amber threshold (0.65)")
plt.axhline(y=0.90, color="#E74C3C", linestyle="--", linewidth=1.5, label="Red threshold (0.90)")
plt.axhline(y=0.45, color="#F39C12", linestyle=":",  linewidth=1.0, label="Uncertainty band lower (0.45)")
plt.axhline(y=0.75, color="#F39C12", linestyle=":",  linewidth=1.0, label="Uncertainty band upper (0.75)")
plt.axvline(x=drift_point, color="black", linestyle="--", linewidth=1.5, label="Drift point")
plt.xlabel("Transaction #")
plt.ylabel("Unified Fraud Score")
plt.title("Fraud Score Distribution with Decision Zones and Uncertainty Flags")
plt.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig("score_zones.png", dpi=150)
plt.show()


In [ ]:
from app.continual_learning.active_learning import ActiveLearningQueue
import asyncio

queue = ActiveLearningQueue(max_size=100)

async def simulate_queue():
    enqueued = 0
    for i, score_val in enumerate(score_values[:100]):
        scores = make_scores(score_val)
        did_enqueue = await queue.maybe_enqueue(
            transaction_ref=f"SIM{i:05d}",
            body={"customer_email": f"user{i}@test.com", "amount": 50000},
            scores=scores,
        )
        if did_enqueue:
            enqueued += 1

    pending = await queue.get_pending_reviews(limit=10)
    return enqueued, pending

enqueued, pending = asyncio.run(simulate_queue())
print(f"\nActive Learning Queue:")
print(f"  Enqueued for review: {enqueued}")
print(f"  Top uncertain (sample):")
for p in pending[:3]:
    print(f"    {p['transaction_ref']} | Score: {p['unified_score']:.3f}")
